[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/jeffery/jeffery.ipynb)

# Jeffery Orbits

Stir a jar of honey with a few grains of rice in it, and the grains do not simply turn with the honey: they line up with the flow, linger, then flip end over end, again and again. A rigid particle cannot stretch with the fluid, only turn, and the fluid presses on it until it turns at the one rate that leaves no torque on it, found by Jeffery in 1922. This notebook writes that rate as one solve on maps, `shape(turning) + turning(shape) == gradient(shape) - shape(gradient.adjoint())`, and lets a sphere, a rod and an ellipsoid with three different axes tumble in a shear flow.

In the index notation of suspension mechanics, in the particle's principal frame, the relative spin reads as $\omega_{ij} - \Omega_{ij} = \frac{a_i^2 - a_j^2}{a_i^2 + a_j^2}\,E_{ij}$, with $\Omega$ the vorticity and $E$ the strain rate of the flow.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.animation import save_animation
from examples.mechanics.jeffery import render
from examples.mechanics.jeffery.core import Tumbling

ga = VGA3D
mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
Stretch = ga.gatype((Vector, Vector))                      # Vector <- Vector: the unit sphere onto the body
Shape = ga.gatype((Vector, Vector))                        # Vector <- Vector: the body's quadric
Gradient = ga.gatype((Vector, Vector))                     # Vector <- Vector: velocity per offset

axes = mv.basis()                                          # [3] Vector
x, y, z = axes                                             # flow, gradient, vorticity
# The bodies: their names and semi-axes; the rod's long semi-axis over its short ones.
NAMES = ["sphere", "rod", "three axes"]
SEMI_AXES = np.array([[1.0, 1.0, 1.0], [4.0, 1.0, 1.0], [4.0, 2.0, 1.0]])
ASPECT = 4.0
# The flow: its velocity along the first axis, per offset along the second.
SHEAR_RATE = 1.0
# The angles of each body's long axis from the vorticity axis at the start, and the start shown moving.
TILTS = np.array([0.15, 0.4, 0.7, 1.0, 1.3])
SHOWN = 2
# The time span in periods of the rod's orbit, the steps per period, and the steps per frame.
PERIODS = 2
STEPS_PER_PERIOD = 120
STRIDE = 3
# The heights at which the flow is drawn.
HEIGHTS = np.linspace(-3.0, 3.0, 7)

## 1. The flow and the bodies

In simple shear, honey between a fixed plate and a sliding one, the velocity along the flow grows with the height: the flow's gradient is a map on offsets. Its curl, the map wedged with the open vector it acts on and contracted, is twice the spin of the fluid. Each body is the unit sphere stretched along its own axes, and its shape is the stretch applied twice: the surface is where `x | shape.solve(x) == 1`. A rotor turns the shape like any map, on its output and its input.

In [ ]:
gradient = SHEAR_RATE * x * (y | Vector)                                       # [] Gradient
fluid_spin = (Vector ^ gradient(Vector)).contract(1, 2) / 2                    # [] Bivector
stretches = (axes * SEMI_AXES * (axes | Vector)).sum(axis=-1)                  # [bodies] Stretch


def placed(stretches: Stretch, rotors: Rotor) -> Shape:
    """The shape of each body, the stretch twice, turned by its rotor."""
    shape = stretches(stretches)
    return rotors >> shape(rotors << Vector)

## 2. Jeffery's spin

Under a spin, a bivector, each offset of the body moves at `offset | spin`: the turning map. Jeffery's spin makes the shape's product with the turning map, from both sides, equal to the gradient's product with the shape less the shape's with the gradient's adjoint. Both sides are maps that turn, so their curls hold all of them: a map from the spin to a bivector on the left, a bivector on the right, and one solve. For a sphere the shape is a multiple of the identity, and the spin is the fluid's.

In [ ]:
def spin(shape: Shape, gradient: Gradient) -> Bivector:
    """The torque-free spin of a rigid ellipsoid of the given shape in a flow of the given gradient."""
    turning = Vector | Bivector                                               # Vector <- (Vector, Bivector)
    left = shape(turning) + turning(shape)                                    # [...] Vector <- (Vector, Bivector)
    right = gradient(shape) - shape(gradient.adjoint())                       # [...] Vector <- Vector
    left_curl = (Vector ^ left(Vector)).contract(1, 2)                        # [...] Bivector <- Bivector
    return left_curl.solve((Vector ^ right(Vector)).contract(1, 2))           # [...] Bivector


sphere_spin = spin(placed(stretches[0], mv.rotor()), gradient)                 # [] Bivector: the fluid's

## 3. Tumbling

Each step turns every body by its spin, taken halfway through the step, and all bodies from all starts step together as one batch. On the sphere of directions, the sphere's axis goes around the vorticity axis at a steady rate. The rod's axis lingers along the flow, flips, and comes back to where it started after Jeffery's period, $2\pi(r + 1/r)$ over the shear rate for a rod of aspect $r$, on a closed orbit. The ellipsoid with three different axes never quite closes its path.

In [ ]:
def step(rotors: Rotor, dt: float) -> Rotor:
    """The bodies turned for a time `dt` by their spin at the midpoint of the step."""
    halfway = (spin(placed(stretches[:, None], rotors), gradient) * (-dt / 4)).exp() * rotors
    return (spin(placed(stretches[:, None], halfway), gradient) * (-dt / 2)).exp() * rotors


def tumble(rotors: Rotor, dt: float, steps: int):
    """The bodies' orientations, from the start and after each step."""
    yield rotors
    for _ in range(steps):
        rotors = step(rotors, dt)
        yield rotors


period = 2 * np.pi * (ASPECT + 1 / ASPECT) / SHEAR_RATE
starts = ((x ^ z) * (-(np.pi / 2 - TILTS) / 2)).exp()[None].broadcast_to((len(NAMES), len(TILTS)))   # [bodies, starts] Rotor
rotors = stack(list(tumble(starts, period / STEPS_PER_PERIOD, PERIODS * STEPS_PER_PERIOD)))           # [steps + 1, bodies, starts] Rotor
trails = rotors >> x                                                           # [steps + 1, bodies, starts] Vector
shown = rotors[::STRIDE, :, SHOWN]                                             # [frames, bodies] Rotor
heights = y * HEIGHTS                                                          # [heights] Vector
tumbling = Tumbling(trails, stretches, shown, shown >> (x * SEMI_AXES[:, 0]), heights, gradient(heights))
render.draw_orbits(tumbling, NAMES);

In [ ]:
display(Image(filename=save_animation(render.animate_tumbling(tumbling, NAMES), "jeffery_tumbling", 60)))

In [ ]:
# checks
# A sphere spins at the fluid's own rate.
np.testing.assert_allclose((sphere_spin - fluid_spin).norm().to_array(), 0.0, atol=1e-14)
# The rod's axis is back where it started after one period, and keeps Jeffery's orbit constant; the
# tolerances test the step.
rod = trails[:, 1]                                                             # [steps + 1, starts] Vector
np.testing.assert_allclose((rod[STEPS_PER_PERIOD] - rod[0]).norm().to_array(), 0.0, atol=2e-2)
flowing, across, spinning = rod | x, rod | y, rod | z                           # [steps + 1, starts] Scalar each
constant = ((flowing.squared() + ASPECT**2 * across.squared()) / (ASPECT**2 * spinning.squared())).to_array()
np.testing.assert_allclose(constant / constant[0], 1.0, rtol=2e-2)